### Goals: 
* Showcase computation of our desired jacobian-vector product (JVP) for a small toy network case using either:
* 1) torch.autograd.functional.jacobian (which has $\mathcal{O}(d^2)$ complexity),
  2) torch.autograd.grad (which has $\mathcal{O}(d)$ complexity using for loops,
  3) using torch.autograd.grad + torch.vmap to avoid loop over batch dimension.
* This is meant as a simple/early example for Bruce to work with.

In [7]:
#general 
import numpy as np
import os, sys
import torch

In [8]:
#specific to vfm repo
#this is mostly so you have a net to work with ... 
sys.path.append('/home/dfd4/vfm_D_min_clipping/') #swap this for path to VFM repo in your local machine 
import dnnlib
from training.networks import ToyMLP

### Construct a network to use 

In [9]:
#set up device first
device_name = 'cuda:1' #can swap this to cuda:0, etc pending on resources
device = torch.device(device_name)

In [10]:
#create ToyMLP instance, 
#adjust it to train mode, tracking grads and pass it to device
mlp = ToyMLP(dim=784, time_varying=True, n_hidden=6, w=64)
mlp.train().to(device)

ToyMLP(
  (net): Sequential(
    (0): Linear(in_features=785, out_features=64, bias=True)
    (1): SELU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): SELU()
    (4): Linear(in_features=64, out_features=64, bias=True)
    (5): SELU()
    (6): Linear(in_features=64, out_features=64, bias=True)
    (7): SELU()
    (8): Linear(in_features=64, out_features=64, bias=True)
    (9): SELU()
    (10): Linear(in_features=64, out_features=64, bias=True)
    (11): SELU()
    (12): Linear(in_features=64, out_features=64, bias=True)
    (13): SELU()
    (14): Linear(in_features=64, out_features=784, bias=True)
  )
)

#### Above yields a simple mlp with $n$ number of hidden layers, each with width $w$.  
#### Note that input feature size is data_dim + 1. This is by design, as this net takes also a time input which is concatenated to flattened image input.
### Let's compute one desired jvp using torch.autograd.functional.jacobian ... 

In [11]:
#first, set up inputs for net 
#am choosing small batch size to make computation faster 
batch_size = 6
flat_data_dim = 784 
imgs = torch.randn(batch_size, flat_data_dim).type(torch.float32).to(device)
ts = torch.rand(batch_size, device=device) 

In [12]:
#ok now calc Jacobian of net_out w.r.t imgs input 
#set requires_grad to True for net inputs... 
ts.requires_grad=True
imgs.requires_grad=True

In [13]:
mlp_jac = torch.autograd.functional.jacobian(mlp, (imgs, ts))

In [14]:
print(len(mlp_jac))
print(mlp_jac[0].shape)
print(mlp_jac[1].shape)

2
torch.Size([6, 784, 6, 784])
torch.Size([6, 784, 6])


In [15]:
print(mlp_jac[0])

tensor([[[[ 1.1224e-03, -2.9081e-04,  2.0930e-03,  ...,  1.5872e-03,
            1.8023e-03, -1.7842e-03],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00]],

         [[ 1.4314e-04, -1.1700e-04,  2.2728e-04,  ..., -3.8570e-03,
           -8.8283e-04, -1.6690e-03],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
            0.0000e+00,  0.0000e+00],
          [ 0.0000e+00,  0.0000e+00

#### Note that torch's Jac method will produce one Jacobian output per each input of the function call. So, in this case, first item in list is Jacobian of net output w.r.t. first (i.e., imgs) input and second item is Jacobian of output w.r.t second input (ts).
#### Note also that Jac ouputs have several FULL ZERO rows... This is because batch items are independent of each other - that is, we can collapse across batch dim in dim==2.

In [17]:
#imgs jvp
torch_jac = torch.sum(mlp_jac[0], dim=2) #collapse over extra batch dim, see comment above 
nabla_imgs = torch_jac.transpose(2,1) #transpose, to get gradient
#compute actual u \cdot nabla product. 
#note that this is between above nabla_imgs and corresponding network output tensor u 
imgs_jvp = torch.einsum('bij, bjk -> bik', mlp(imgs, ts).unsqueeze(1), nabla_imgs).squeeze(1) #bs, dim 

In [18]:
print(imgs_jvp)
print(imgs_jvp.shape)

tensor([[ 6.8849e-03,  1.7946e-02,  9.3102e-03,  ..., -4.8871e-03,
         -2.4615e-03, -2.0011e-03],
        [ 3.8782e-03,  2.4920e-02,  1.2851e-02,  ...,  4.0897e-03,
          2.8487e-03,  2.8590e-03],
        [ 4.2174e-03,  1.0268e-02,  9.8640e-03,  ..., -2.4153e-03,
         -8.5127e-03, -9.9371e-03],
        [ 6.4965e-03,  1.4534e-02,  1.0329e-02,  ...,  2.9265e-03,
          9.6130e-04,  2.5810e-05],
        [ 2.3164e-03,  1.3555e-02,  1.9556e-02,  ..., -4.7026e-03,
         -1.7228e-03, -1.0633e-02],
        [ 7.9494e-03,  1.9257e-02,  1.4904e-02,  ..., -1.9011e-03,
         -3.4706e-03, -6.2936e-03]], device='cuda:1',
       grad_fn=<SqueezeBackward1>)
torch.Size([6, 784])


#### Let's de-compress the above a bit. First, we take Jacobian of net output w.r.t to its first input (imgs) and sum/collapse it over dimension 2, since each output is only dependent on its corresponding batch input. 
#### Then, we take transpose w.r.t to 2 last dimensions - this is d/t relationship between Jacobian and gradients... Typically, for scalar functions, gradients are column vectors (i.e., rows of Jac transposed)
#### Will triple check with John that this is needed here... 

#### Finally, we use an Einstein summation to compute the desired product over the batch -- this yields the $u \cdot \nabla $ product we want for the Lie derivative calculation.

#### Couple of points here: 
* 1) In actual code, this will take LONGER to run, because in there imgs are not just simple inputs, but instead interpolations between original data and outputs of another, separate encoder network.
  2) Additionally, we compute two such JVPs (one for flow net, another for dynamics net) and compute partial derivative w.r.t. to time argument (ts) as well.
  3) All of these then go into computing out Lie derivative loss: $\mathcal{L}_{Lie} = \partial_{\tau} \mathbf{v} + \mathbf{u} \cdot \nabla_{\mathbf{v}} - \mathbf{v} \cdot \nabla_{\mathbf{u}}$

### Ok, let's compute same jvp now with torch.grad.autograd

In [81]:
#small method to compute desired Jacobian, for a batch 
def batch_jacobian(model, imgs, ts):
    """Computes the Jacobian of a batch of outputs w.r.t a batch of inputs."""

    batch_size, input_size = imgs.shape
    output_size = model(imgs, ts).shape[1]

    jacobian = torch.zeros(batch_size, output_size, input_size)

    #note that we loop over batch AND dimensions here! 
    for i in range(batch_size):
        for j in range(output_size):
            grad_outputs = torch.zeros_like(model(imgs, ts))
            grad_outputs[i, j] = 1.0
            jacobian[i, j] = torch.autograd.grad(
                model(imgs, ts), imgs, grad_outputs=grad_outputs, retain_graph=True
            )[0][i]

    return jacobian

In [82]:
ag_jac = batch_jacobian(mlp, imgs, ts)

In [83]:
#check that shape matches - this should already be collapsed across extra batch dim
print(ag_jac.shape)

torch.Size([6, 784, 784])


In [84]:
#check that this produces correct/desired Jac 
np.allclose(ag_jac.detach().cpu().numpy(), torch_jac.detach().cpu().numpy())

True

In [23]:
#ok, now compute jvp 
nabla_imgs_ag = ag_jac.transpose(2,1) #transpose to get grad 
imgs_jvp_ag = torch.einsum('bij, bjk -> bik', mlp(imgs, ts).unsqueeze(1), nabla_imgs_ag.to(device)).squeeze(1) #bs, dim 

In [24]:
#check that these results are indeed the same... 
np.allclose(imgs_jvp.detach().cpu().numpy(), imgs_jvp_ag.detach().cpu().numpy())

True

### Ok, now let's use torch vmap and torch.autograd.grad to run the above 
#### This avoids loop over batch items but comes at cost of larger VRAM requirements...

In [36]:
get_jvp = lambda v: torch.autograd.grad(mlp(imgs, ts), imgs, v, retain_graph=True)

In [37]:
#method to create v vector we will need to vmap over batch elements 
def build_IN_vmap(shape):
    """
    Computes IN we will use for vmapping 
    over torch.autograd.grad call 
    """
    I_N = []
    for i in range(shape[0]):
        for j in range(shape[1]):
            curr_IN = torch.zeros(shape)
            curr_IN[i, j]=1
            I_N.append(curr_IN)
    I_N = torch.cat(I_N, dim=0)
    return I_N.reshape((-1, shape[0], shape[1]))

In [38]:
vmap_v = build_IN_vmap((6, 784))

In [39]:
#check that shape matches - should be [bs*dim, bs, dim]
vmap_v.shape

torch.Size([4704, 6, 784])

In [49]:
vmap_ag_jac = torch.vmap(get_jvp)(vmap_v.type(torch.float32).to(device))[0]

In [51]:
#check that shape is correct - should be [bs*dim, bs, dim]
print(vmap_ag_jac.shape)
#reshape this output to [bs, dim, bs, dim]
#collapse over dim==2 as before 
vmap_ag_jac = vmap_ag_jac.reshape(batch_size, imgs.shape[1], batch_size, imgs.shape[1])
vmap_ag_jac = vmap_ag_jac.sum(2)
print(vmap_ag_jac.shape)

torch.Size([4704, 6, 784])
torch.Size([6, 784, 784])


In [52]:
#ok now check that this is indeed identical to previous results obtained with loop + AG and Jac 
vmap_ag_jac.all() == ag_jac.all()

tensor(True, device='cuda:1')

In [54]:
vmap_ag_jac.all() == torch_jac.all()

tensor(True, device='cuda:1')

### General Comments: 
1) I need to check with John if indeed I need to transpose Jac here? This is how I implemented things originally and this matches results computed by hand.
2) All cases above still end up computing a Jacobian different ways and then doing the Jacobian vector product... This is in part due to difficulty of finding a proper vector $v$ that would correctly select ONLY desired row of Jac and ALSO multiply it by our corresponding output vector.
3) I don't think this is possible really? But discuss it with John and then update Bruce 